# 3.4 · Observa y depura tu agente con trazas
**Módulo 3 · Coordina, fundamenta y planea** — *Seguridad de agentes con OpenShell y NemoClaw* (Platzi × NVIDIA Deep Learning Institute)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uncrayon/curso-seguridad-agentes-nemoclaw/blob/main/clases/3.4-observabilidad-y-trazas/3.4-observabilidad-y-trazas.ipynb)

Tu agente ya enruta, busca en la FAQ y lanza subagentes. Cuando algo sale mal, `print` ya no alcanza: no sabes **qué paso** falló, **con qué argumentos**, **cuánto tardó** ni **de dónde salió** una cita. En este notebook le pones **trazas** a tu agente de soporte:

1. **Instrumentar** el loop con OpenTelemetry: un *span* por el agente, por cada llamada al modelo y por cada herramienta.
2. **Ver** la traza como árbol aquí mismo y, si lo tienes corriendo, en **Phoenix**.
3. **Depurar** tres errores reales solo con la traza: una herramienta que falla, un loop atascado y una cita inventada. Y encontrar el paso lento.
4. **Proteger** la traza: tachar las keys antes de guardarla, porque una traza guarda prompts completos.

Lo aplicas al proyecto **Soporte NemoClaw**: cada respuesta del agente queda con su traza, lista para adjuntar a un reporte de error.

*Clase nueva de la adaptación de Platzi: no tiene lección equivalente en el curso de NVIDIA DLI. Usa estándares abiertos (OpenTelemetry, OpenInference) y Arize Phoenix, que también es uno de los destinos del NeMo Agent Toolkit de NVIDIA.*

> **¿Sin API key?** Ejecuta todo igual: el notebook detecta que no hay key y usa un **modelo simulado (MOCK)** que imita la forma de las respuestas de la API. Con tu key `nvapi-` (clase 1.2), las mismas celdas llaman al modelo real.

> **¿No programas?** No necesitas leer el código. Presiona ▶ en cada celda, en orden, y lee el texto y lo que aparece debajo (o usa **Entorno de ejecución → Ejecutar todo** en Colab). Las celdas con ⚙️ son de preparación: se ejecutan igual, pero no necesitas abrirlas. Las palabras nuevas están en el [glosario](https://github.com/uncrayon/curso-seguridad-agentes-nemoclaw/blob/main/GLOSARIO.md).

In [ ]:
# @title ⚙️ Configuración (ejecútala siempre primero)
# --- Configuración (ejecútala siempre primero) ---------------------------------
import os, sys, json, subprocess

def _instala(paquete, modulo=None):
    try:
        __import__(modulo or paquete)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])

_instala("openai")
_instala("python-dotenv", "dotenv")

# 1) Google Colab: guarda tu key en Secrets (icono de llave 🔑) con el nombre NVIDIA_API_KEY
try:
    from google.colab import userdata
    os.environ.setdefault("NVIDIA_API_KEY", userdata.get("NVIDIA_API_KEY") or "")
except Exception:
    pass
# 2) Local: archivo .env en la raíz del repo (copia .env.example) o `export NVIDIA_API_KEY=nvapi-...`
from dotenv import load_dotenv
load_dotenv(); load_dotenv("../../.env")

BASE_URL    = os.environ.get("NVIDIA_BASE_URL", "https://integrate.api.nvidia.com/v1")
MODEL       = os.environ.get("NVIDIA_MODEL", "nvidia/nemotron-3.5-lightning-30b-a3b")
EMBED_MODEL = os.environ.get("NVIDIA_EMBED_MODEL", "nvidia/nemotron-3-embed-1b")
API_KEY     = (os.environ.get("NVIDIA_API_KEY") or "").strip()
USE_MOCK    = not API_KEY

if API_KEY and "integrate.api.nvidia.com" in BASE_URL and not API_KEY.startswith("nvapi-"):
    print("⚠  Tu key no empieza con 'nvapi-'. ¿Copiaste la key completa de build.nvidia.com?")
print("Modo:", "MOCK (sin NVIDIA_API_KEY: respuestas simuladas)" if USE_MOCK else f"EN VIVO · {MODEL}")

In [ ]:
# @title ⚙️ Cliente: modelo real o simulado, con errores explicados en español
# --- Cliente: modelo real o simulado, con errores explicados en español --------
import re, time, math, hashlib
from types import SimpleNamespace as NS

def _msg(content):
    m = NS(content=content, tool_calls=None, role="assistant", reasoning_content=None)
    return NS(choices=[NS(message=m, finish_reason="stop")], usage=NS(prompt_tokens=0, completion_tokens=0, total_tokens=0))

def _call(name, args):
    tc = NS(id="call_" + hashlib.md5((name + json.dumps(args)).encode()).hexdigest()[:8], type="function",
            function=NS(name=name, arguments=json.dumps(args, ensure_ascii=False)))
    m = NS(content=None, tool_calls=[tc], role="assistant", reasoning_content=None)
    return NS(choices=[NS(message=m, finish_reason="tool_calls")], usage=NS(prompt_tokens=0, completion_tokens=0, total_tokens=0))

class MockClient:
    """Imita la FORMA de la API de chat completions. No es inteligente: sigue reglas.
    Cada notebook puede agregar reglas propias con  @regla_mock  (ver más abajo)."""
    reglas = []
    def __init__(self):
        self.chat = NS(completions=NS(create=self._create))
        self.embeddings = NS(create=self._embed)
    def _create(self, model=None, messages=None, tools=None, **kw):
        nombres = [t["function"]["name"] for t in (tools or [])]
        for r in self.reglas:
            out = r(messages, nombres)
            if out is not None:
                return out
        ultimo = messages[-1]
        texto = (ultimo.get("content") or "")
        if ultimo["role"] == "tool":
            return _msg("Según la herramienta: " + str(ultimo["content"]))
        if re.search(r"c[oó]mo me llamo|what is my name", texto, re.I):
            vistos = [re.search(r"(?:me llamo|my name is)\s+(\w+)", m.get("content") or "", re.I)
                      for m in messages[:-1] if m["role"] == "user"]
            vistos = [v.group(1) for v in vistos if v]
            return _msg(f"Te llamas {vistos[-1]}." if vistos else "No sé cómo te llamas: no me lo has dicho en esta conversación.")
        return _msg("(mock) Respuesta simulada. Configura NVIDIA_API_KEY para hablar con el modelo real.")
    def _embed(self, model=None, input=None, **kw):
        textos = [input] if isinstance(input, str) else list(input)
        return NS(data=[NS(embedding=_vector_bolsa(t), index=i) for i, t in enumerate(textos)])

def regla_mock(f):
    """Decorador: agrega una regla al modelo simulado. f(messages, nombres_de_tools) -> respuesta | None"""
    MockClient.reglas.append(f)
    return f

_VACIAS = set("que qué como cómo para por con una uno los las del este esta eso esto hago hace tengo puedo sale mis tus sus the and you what how".split())

def _vector_bolsa(texto, dim=256):
    """Embedding simulado: bolsa de palabras con hashing. Sirve para que RAG funcione offline."""
    v = [0.0] * dim
    for w in re.findall(r"\w+", texto.lower()):
        if len(w) > 2 and w not in _VACIAS:
            w = w[:6]                       # raíz burda: "reanudar" ~ "reanudé"
            v[int(hashlib.md5(w.encode()).hexdigest(), 16) % dim] += 1.0
    n = math.sqrt(sum(x * x for x in v)) or 1.0
    return [x / n for x in v]

def diagnostico(e):
    """Traduce un error de la API a: qué pasó + qué hacer. Es el 'FAQ' en código."""
    status = getattr(e, "status_code", None)
    cuerpo = str(getattr(e, "body", "") or e)[:300]
    tabla = {
        400: "400 · Solicitud inválida: algún parámetro no le gusta al modelo (¿tools en un modelo que no las soporta? ¿max_tokens muy alto?). Lee el cuerpo del error.",
        401: "401 · Key rechazada: revisa que NVIDIA_API_KEY sea la key completa (empieza con nvapi-), sin espacios ni comillas. Si la regeneraste, la vieja ya no sirve.",
        403: "403 · Tu key es válida pero no tiene acceso a ese modelo o se acabaron tus créditos. Revisa tu cuenta en build.nvidia.com o prueba otro modelo.",
        404: "404 · Ese modelo o esa ruta no existen. Revisa MODEL / EMBED_MODEL (copia el ID exacto del catálogo) y que BASE_URL termine en /v1.",
        410: "410 · El modelo fue retirado del endpoint gratuito. Cambia MODEL por uno vigente del catálogo de build.nvidia.com.",
        422: "422 · Argumentos con formato incorrecto (p. ej. falta input_type en embeddings o un esquema de tool inválido).",
        429: "429 · Demasiadas solicitudes por minuto. Espera unos segundos y reintenta; no ejecutes celdas en ráfaga.",
    }
    if status in tabla:
        return f"{tabla[status]}\n   Detalle del servidor: {cuerpo}"
    if status and 500 <= status < 600:
        return f"{status} · Falla del lado del servidor o el modelo está arrancando. Reintenta en 30-60 s; si persiste, prueba otro modelo.\n   Detalle: {cuerpo}"
    nombre = type(e).__name__
    if "Timeout" in nombre:
        return "Timeout · El modelo tardó demasiado. Reintenta, baja max_tokens o usa un modelo más rápido."
    if "Connection" in nombre:
        return "Sin conexión · No se pudo llegar a BASE_URL. Revisa tu internet, VPN/proxy corporativo o el valor de BASE_URL."
    return f"{nombre}: {cuerpo}"

if USE_MOCK:
    client = MockClient()
else:
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=60, max_retries=2)

def chat(messages, tools=None, model=None, **kw):
    """Una llamada al modelo. Devuelve la respuesta cruda (choices[0].message, finish_reason, usage)."""
    kw.setdefault("max_tokens", 4096)          # los modelos que razonan gastan tokens antes de responder
    if tools:
        kw["tools"] = tools
    try:
        r = client.chat.completions.create(model=model or MODEL, messages=messages, **kw)
    except Exception as e:
        raise RuntimeError(diagnostico(e)) from None
    ch = r.choices[0]
    if ch.finish_reason == "length" and not (ch.message.content or ch.message.tool_calls):
        print("⚠  finish_reason=length sin contenido: el modelo gastó max_tokens razonando. Sube max_tokens.")
    return r

def embed(textos, input_type="passage", model=None):
    """Embeddings. input_type='query' para preguntas, 'passage' para documentos."""
    textos = [textos] if isinstance(textos, str) else list(textos)
    try:
        r = client.embeddings.create(model=model or EMBED_MODEL, input=textos,
                                     extra_body={"input_type": input_type, "truncate": "END"}) if not USE_MOCK \
            else client.embeddings.create(model=model or EMBED_MODEL, input=textos)
    except Exception as e:
        raise RuntimeError(diagnostico(e)) from None
    return [d.embedding for d in r.data]

print("Cliente listo:", "MockClient" if USE_MOCK else BASE_URL)

In [ ]:
# @title ⚙️ Ayudas de este módulo (ejecútala después de las dos celdas anteriores)
# --- Ayudas de este módulo (ejecútala después de las dos celdas anteriores) -------
import json, re, time
from concurrent.futures import ThreadPoolExecutor, as_completed

LLAMADAS = []          # registro de cada llamada al modelo: (rol, segundos). Así contamos el costo.
LATENCIA_MOCK = 0.3    # sin key simulamos lo que tarda una llamada (en vivo suele ser de 1 a 10 s)

def llamar(rol, mensajes, **kw):
    """Una llamada al modelo, registrada. Devuelve la respuesta completa (choices[0].message...)."""
    t0 = time.time()
    if USE_MOCK:
        time.sleep(LATENCIA_MOCK)
    kw.setdefault("temperature", 0)            # clasificar y planear: queremos respuestas estables
    r = chat(mensajes, **kw)
    LLAMADAS.append((rol, round(time.time() - t0, 2)))
    return r

def llm(rol, usuario, sistema="", **kw):
    """Atajo: system + user -> texto. El rol va en la primera línea del system (y en el registro)."""
    mensajes = [{"role": "system", "content": f"ROL: {rol}\n{sistema}".strip()},
                {"role": "user", "content": usuario}]
    return llamar(rol, mensajes, **kw).choices[0].message.content or ""

def extraer_json(texto):
    """Los modelos a veces envuelven el JSON en prosa o en ```json. Tomamos el primer {...}."""
    m = re.search(r"\{.*\}", texto or "", re.S)
    if not m:
        raise ValueError(f"no encontré JSON en: {(texto or '')[:80]!r}")
    return json.loads(m.group(0))

class ErrorSimulado(Exception):
    """Imita un error HTTP de la API (status_code + body) para practicar el diagnóstico sin gastar llamadas."""
    def __init__(self, status_code, body="(error simulado)"):
        super().__init__(f"HTTP {status_code}")
        self.status_code, self.body = status_code, body

def _rol(messages):
    s = messages[0]["content"] if messages and messages[0]["role"] == "system" else ""
    return s.split("\n")[0].replace("ROL:", "").strip()

def _ultimo_usuario(messages):
    return next((m["content"] for m in reversed(messages) if m["role"] == "user"), "") or ""

def _primeras(texto, n=2):
    """Las primeras n frases de un texto (para respuestas simuladas cortas)."""
    frases = re.split(r"(?<=[.!?])\s+", texto.strip())
    return " ".join(frases[:n])

print("Ayudas listas: llm(), llamar(), extraer_json(), ErrorSimulado")

In [ ]:
# @title ⚙️ Reglas del modelo simulado para ESTE notebook (solo se usan sin API key)
# --- Reglas del modelo simulado para ESTE notebook (solo se usan sin API key) -----
# No necesitas leerlas. Imitan a un agente de soporte con herramientas y, a propósito,
# reproducen los tres errores que vas a cazar con las trazas: herramienta que falla,
# loop atascado y cita inventada.
_ID = re.compile(r"\[([\w.\-]+\.md#[\w\-]+)\]")

def _ids_en(textos):
    vistos = []
    for t in textos:
        for i in _ID.findall(t or ""):
            if i not in vistos:
                vistos.append(i)
    return vistos

def _primera_respuesta(texto):
    m = re.search(r"\nR: (.+)", texto or "")
    return _primeras(m.group(1), 2) if m else ""

@regla_mock
def _mock_soporte_trazado(messages, nombres):
    if _rol(messages) != "soporte-trazado":
        return None
    pregunta = next((m["content"] for m in messages if m["role"] == "user"), "")
    p = pregunta.lower()
    salidas = [m["content"] for m in messages if m["role"] == "tool"]
    n = len(salidas)
    # Caso loop: una pregunta que la FAQ no cubre; el "modelo" repite la misma búsqueda
    if "factura" in p:
        return _call("buscar_faq", {"consulta": "factura de Brev en pesos mexicanos", "k": 3})
    # Caso cita inventada: responde con una cita real y una que no salió de ninguna herramienta
    if "gratis para siempre" in p:
        if n == 0:
            return _call("buscar_faq", {"consulta": "créditos de Brev gratis launchable", "k": 2})
        reales = _ids_en(salidas)[:1]
        citas = " ".join(f"[{i}]" for i in reales + ["brev.md#la-gpu-es-gratis-para-siempre"])
        return _msg("**Qué pasa:** el launchable incluye GPU gratis para siempre, sin límite de horas.\n"
                    f"**Fuente:** {citas}")
    # Caso laboratorio: primero revisa la instancia y luego busca en la FAQ
    sobre_lab = any(w in p for w in ("laboratorio", "instancia", "brev"))
    if sobre_lab and n == 0:
        return _call("estado_laboratorio", {"instancia": "mi-lab"})
    if n == (1 if sobre_lab else 0):
        # Un buen agente reformula la búsqueda con lo que ya observó
        if sobre_lab:
            consulta = "cookie de acceso vencida al reanudar el launchable"
        elif "401" in p:
            consulta = "error 401 key rechazada notebook"
        else:
            consulta = pregunta[:90]
        return _call("buscar_faq", {"consulta": consulta, "k": 3})
    # Respuesta final: lo que dijeron las herramientas, con sus citas
    ids = _ids_en(salidas)[:2]
    estado = next((s for s in salidas if s.startswith("{") or s.startswith("ERROR")), "")
    partes = []
    if estado.startswith("ERROR"):
        partes.append("No pude verificar el estado de tu instancia (la herramienta falló).")
    elif "vencida" in estado:
        partes.append("Tu instancia está RUNNING, pero la cookie de sesión del laboratorio venció al reanudar.")
    if ids:
        partes.append("**Qué pasa:** " + _primera_respuesta(next(s for s in salidas if "[" in s)))
        partes.append("**Fuente:** " + " ".join(f"[{i}]" for i in ids))
    else:
        partes.append("La FAQ no cubre esto.")
    return _msg("\n".join(partes))

print("Reglas mock de la clase 3.4 listas")

## 1 · Por qué `print` no alcanza
Un agente toma decisiones en varios pasos. Cuando la respuesta final está mal, la causa puede estar en cualquiera:

| Pregunta al depurar | Dónde está la respuesta |
|---|---|
| ¿Qué herramienta llamó y con qué argumentos? | Span de la herramienta (`TOOL`) |
| ¿Qué le mandamos al modelo y qué contestó? | Span de la llamada (`LLM`) |
| ¿Qué fragmentos recuperó la búsqueda y con qué puntaje? | Span de recuperación (`RETRIEVER`) |
| ¿Qué paso tardó más? ¿Cuántos tokens gastó? | Duración y atributos de cada span |
| ¿Dónde se rompió? | El span marcado con **error** |

Vocabulario mínimo:
- **Traza:** todo lo que pasó para responder **una** pregunta. Es un árbol.
- **Span:** una operación dentro de la traza, con inicio, fin, atributos y estado (ok o error). Cada span tiene un padre, salvo la raíz.
- **OpenTelemetry (OTel):** el estándar abierto para producir y enviar trazas. No depende de ningún proveedor.
- **OpenInference:** convenciones sobre OTel para aplicaciones de IA: `openinference.span.kind` = `AGENT`, `LLM`, `TOOL`, `RETRIEVER`, `CHAIN`...
- **Phoenix** (Arize, código abierto): una interfaz que recibe trazas OTel y las muestra como árbol, con filtros y métricas.

## 2 · La FAQ y la herramienta de búsqueda (clases 3.2 y 3.3)

In [ ]:
# @title ⚙️ La FAQ del curso: muestra incrustada + carga completa si tienes el repo
# --- La FAQ del curso: muestra incrustada + carga completa si tienes el repo ------
# Formato (igual que datos/faq/*.md):  "# Tema"  y luego entradas  "## P: pregunta"  + respuesta.
import pathlib

# Muestra: 14 entradas copiadas de datos/faq/ (mismos archivos y preguntas, para que las citas coincidan).
FAQ_MUESTRA = {
"api-key.md": """# API key de NVIDIA Build

## P: ¿Dónde consigo mi API key de NVIDIA para el curso?
Entra a https://build.nvidia.com e inicia sesión o crea tu cuenta.
Abre tu perfil, ve a **API Keys** y pulsa **Generate API Key**.
Copia la key completa: empieza con `nvapi-`.
Guárdala en un gestor de contraseñas y nunca la pegues en el código.

## P: ¿Dónde guardo mi key y con qué nombre de variable?
Es la misma key `nvapi-` en todo el curso.
En Colab, guárdala en Secrets (icono de llave) como `NVIDIA_API_KEY` y activa el acceso del notebook.
En tu máquina, ponla en el archivo `.env` de la raíz del repo como `NVIDIA_API_KEY` (ese archivo ya está en `.gitignore`).
Para NemoClaw instalado en tu máquina, usa la variable `NVIDIA_INFERENCE_API_KEY`.
""",
"errores-api.md": """# Errores de la API del modelo

## P: ¿Qué significa el error 401 Unauthorized?
Tu key no es válida: está incompleta, tiene espacios o comillas, o la revocaste o regeneraste.
Copia otra vez la key completa (`nvapi-…`) y confirma que tu código lee la variable correcta, por ejemplo con `echo ${NVIDIA_API_KEY:0:6}`, que debe imprimir `nvapi-`.
Reintentar o esperar más no arregla una key inválida.

## P: ¿Por qué me da 403 Forbidden si mi key es válida?
Un 403 de la API significa que tu key no tiene acceso a ese modelo o tu cuenta no tiene cuota.
Prueba con el modelo del curso, `nvidia/nemotron-3.5-lightning-30b-a3b`, y revisa tus créditos en build.nvidia.com.
Si el 403 aparece dentro del sandbox como `CONNECT tunnel failed, response 403`, no es la API: es la política de red de OpenShell.

## P: ¿Qué hago si recibo 429 Too Many Requests?
Mandaste demasiadas solicitudes por minuto.
Espera unos segundos y reintenta una sola vez, respetando el encabezado `Retry-After` si viene.
Pon un límite de pasos al loop del agente y agrega una pausa (`time.sleep()`) entre llamadas en pruebas masivas.
""",
"brev.md": """# Brev y el launchable de NemoClaw

## P: ¿Me cobran si detengo la instancia? ¿Y si la borro?
Encendida (Running), Brev cobra cómputo y disco.
Detenida (Stopped), no cobra cómputo pero sí sigue cobrando el disco, y tus datos se conservan (excepto `/tmp`).
Borrada (Deleted), no cobra nada, pero los datos se pierden para siempre.
Al terminar cada clase, detén la instancia; si no vas a volver en días, bórrala después de guardar tu trabajo en git.

## P: ¿Cómo detengo o borro mi instancia de Brev?
En la consola: brev.nvidia.com, sección GPUs, tu instancia, botón Stop o Delete.
En la CLI: `brev ls` para ver el nombre, `brev stop <instancia>` para detenerla o `brev delete <instancia>` para borrarla, y otra vez `brev ls` para confirmar.
Si `brev stop` responde `does not support stop`, esa instancia solo se puede borrar.

## P: Detuve y volví a encender la instancia y el agente no responde
Tras reiniciar la VM, Docker, el gateway de OpenShell y el sandbox pueden no arrancar solos.
Entra con `brev shell <instancia>` y ejecuta `sudo systemctl start docker`, luego `nemoclaw <sandbox> status`, `nemoclaw <sandbox> start` si quedó detenido y `nemoclaw <sandbox> recover` si el gateway no responde.
""",
"conexion-laboratorio.md": """# Conexión del curso de NVIDIA con el laboratorio

## P: ¿Qué es la "Access session" y de dónde saco la cookie?
Es el valor de la cookie de inicio de sesión de tu launchable.
Abre el launchable en el mismo navegador e inicia sesión.
En DevTools ve a Application, Storage, Cookies, y copia solo el Value completo de la cookie que corresponde: `CF_Authorization` para `brevlab.com`, `_pomerium` para `apps.run.brev.nvidia.com` o `__Host-skybridge-brev-prd` para `gobrev.dev`.
Es una credencial: no la compartas.

## P: Me sale "session expired or unreachable" o "no challenge arrived within 30s"
Tu cookie de acceso venció o el launchable se detuvo.
Confirma en Brev que la instancia está RUNNING, abre el launchable e inicia sesión otra vez, copia un valor nuevo de la cookie y pégalo en Access session.
Lo mismo aplica si ves "Expected JSON but received an HTML page" o "The launchable needs a fresh browser session".
""",
"openshell-politicas.md": """# OpenShell y políticas del sandbox

## P: ¿Qué significa "CONNECT tunnel failed, response 403" dentro del sandbox?
No es un problema de certificados: la red de salida del sandbox está bloqueada por defecto y el proxy de OpenShell rechazó la conexión porque ninguna política la permite.
Mira la línea DENIED con `nemoclaw <sandbox> logs --tail 50`, revisa los presets con `nemoclaw <sandbox> policy list` y agrega el que falte con `nemoclaw <sandbox> policy add <preset>`.

## P: ¿Cómo veo la política activa y agrego un preset como pypi o github?
Ve los presets con `nemoclaw <sandbox> policy list` y la política efectiva completa con `openshell policy get --full <sandbox>`.
Prueba un preset con `nemoclaw <sandbox> policy add pypi --dry-run` y aplícalo con `--yes`; quítalo con `nemoclaw <sandbox> policy remove pypi --yes`.
GitHub no está en la política base: agrégalo a propósito.
Los cambios de red se aplican sin reiniciar.
""",
"seguridad.md": """# Seguridad de keys, cookies y accesos

## P: ¿Dónde guarda NemoClaw mis keys? ¿El agente puede verlas?
NemoClaw registra las keys de los proveedores en el gateway de OpenShell, del lado del host, y los sandboxes nuevos usan proveedores adjuntos que dejan la credencial en el host.
Por eso no pongas keys en `openclaw.json` ni en archivos del workspace: lo que está dentro del sandbox, el agente lo puede leer.
""",
"instalacion-local.md": """# NemoClaw instalado en tu máquina

## P: ¿Qué necesito para instalar NemoClaw en mi computadora?
Mínimo 4 vCPU, 8 GB de RAM y 20 GB de disco libre (recomendado 16 GB y 40 GB).
Necesitas Node.js 22.19 o mayor y Docker funcionando antes de instalar.
Funciona en Ubuntu 24.04, macOS con Apple Silicon y Windows con WSL2; Mac con Intel no está soportado.
Revisa tu máquina con `nemoclaw host probe` e instala con `curl -fsSL https://www.nvidia.com/nemoclaw.sh | bash`.
""",
}

def cargar_faq(carpeta="../../datos/faq"):
    """Devuelve ({archivo: texto}, origen). Usa datos/faq/ si existe (corres dentro del repo); si no, la muestra."""
    p = pathlib.Path(carpeta)
    archivos = [a for a in sorted(p.glob("*.md")) if a.name.lower() != "readme.md"] if p.is_dir() else []
    faq = {a.name: a.read_text(encoding="utf-8") for a in archivos}
    faq = {k: v for k, v in faq.items() if "## P:" in v}
    if faq:
        return faq, f"datos/faq completa ({len(faq)} archivos)"
    return dict(FAQ_MUESTRA), f"muestra incrustada ({len(FAQ_MUESTRA)} archivos)"

def _slug(texto, max_palabras=6):
    return "-".join(re.findall(r"\w+", texto.lower())[:max_palabras])

def trocear(archivo, texto):
    """Un fragmento por cada entrada '## P:'. El tema ('# ...') viaja con cada fragmento."""
    m = re.search(r"^# (.+)$", texto, re.M)
    tema = m.group(1).strip() if m else archivo
    out = []
    for bloque in re.split(r"^## P:\s*", texto, flags=re.M)[1:]:
        pregunta, _, respuesta = bloque.partition("\n")
        pregunta, respuesta = pregunta.strip(), respuesta.strip()
        if pregunta and respuesta and not pregunta.startswith("<"):     # "<...>" = plantilla, no entrada real
            out.append({"id": f"{archivo}#{_slug(pregunta)}", "archivo": archivo, "tema": tema,
                        "pregunta": pregunta, "respuesta": respuesta})
    return out

def fragmentos_de(faq):
    """Trocea todos los archivos y garantiza ids únicos (para poder citarlos)."""
    vistos, out = {}, []
    for archivo, texto in faq.items():
        for f in trocear(archivo, texto):
            n = vistos.get(f["id"], 0)
            vistos[f["id"]] = n + 1
            if n:
                f["id"] += f"-{n + 1}"
            out.append(f)
    return out

FAQ, ORIGEN = cargar_faq()
FRAGMENTOS = fragmentos_de(FAQ)
print(f"FAQ: {ORIGEN} · {len(FRAGMENTOS)} entradas")
for a in FAQ:
    print(f"  {a:18s} {sum(f['archivo'] == a for f in FRAGMENTOS)} entradas")

In [ ]:
# @title ⚙️ El índice RAG de la clase 3.2, en una celda
# --- El índice RAG de la clase 3.2, en una celda -----------------------------------
import math

MIN_SCORE = 0.10 if USE_MOCK else 0.30     # umbral de evidencia [VERIFICAR EN LAB el valor en vivo]

def texto_para_indexar(f):
    return f"{f['tema']}. P: {f['pregunta']}\n{f['respuesta']}"

def coseno(a, b):
    num = sum(x * y for x, y in zip(a, b))
    den = math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b))
    return num / den if den else 0.0

class Indice:
    def __init__(self, fragmentos, lote=16):
        self.fragmentos, self.vectores = fragmentos, []
        textos = [texto_para_indexar(f) for f in fragmentos]
        for i in range(0, len(textos), lote):
            self.vectores += embed(textos[i:i + lote], input_type="passage")
    def buscar(self, consulta, k=3, minimo=0.0):
        q = embed(consulta, input_type="query")[0]
        puntajes = sorted(((coseno(q, v), f) for v, f in zip(self.vectores, self.fragmentos)), key=lambda x: -x[0])
        return [{"score": round(s, 3), **f} for s, f in puntajes[:k] if s >= minimo]

def armar_contexto(hits):
    return "\n\n".join(f"[{h['id']}] P: {h['pregunta']}\nR: {h['respuesta']}" for h in hits)

def buscar_faq(consulta, k=3):
    """La herramienta RAG: devuelve fragmentos con su id, o SIN_EVIDENCIA."""
    if not isinstance(consulta, str) or not consulta.strip():
        return "ERROR: 'consulta' debe ser texto no vacío"
    if not isinstance(k, int) or not 1 <= k <= 5:
        return "ERROR: 'k' debe ser un entero de 1 a 5"
    hits = INDICE.buscar(consulta, k=k, minimo=MIN_SCORE)
    return armar_contexto(hits) if hits else "SIN_EVIDENCIA: ningún fragmento supera el umbral."

INDICE = Indice(FRAGMENTOS)
IDS_FAQ = {f["id"] for f in FRAGMENTOS}
print(f"Índice listo: {len(INDICE.vectores)} vectores de {len(INDICE.vectores[0])} dimensiones")
print(buscar_faq("me sale 401 en el laboratorio", k=1)[:160], "...")

## 3 · Configura el tracer
Usamos el SDK de OpenTelemetry con **dos destinos**:
- **En memoria** (siempre): para dibujar la traza aquí y analizarla con código. No necesitas nada más.
- **Phoenix** (si lo encuentra): exporta por OTLP/HTTP a `http://localhost:6006/v1/traces` o a la URL de `PHOENIX_COLLECTOR_ENDPOINT`.

Para levantar Phoenix en tu máquina, en otra terminal: `pip install arize-phoenix` y `phoenix serve` (o `docker run -p 6006:6006 arizephoenix/phoenix:latest`). Luego vuelve a ejecutar esta celda. En Colab, deja que use solo la memoria: todo el notebook funciona igual.

In [ ]:
# --- Tracer de OpenTelemetry: memoria siempre, Phoenix si está disponible -----------
_instala("opentelemetry-sdk", "opentelemetry.sdk")
_instala("opentelemetry-exporter-otlp-proto-http", "opentelemetry.exporter.otlp.proto.http")
import socket
from urllib.parse import urlparse
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor, BatchSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.trace import Status, StatusCode

PROYECTO = "soporte-nemoclaw"

def _phoenix_url():
    """URL del colector de Phoenix, o None si no hay uno escuchando."""
    url = os.environ.get("PHOENIX_COLLECTOR_ENDPOINT", "").strip()
    if url:
        return url.rstrip("/") + ("" if url.rstrip("/").endswith("/v1/traces") else "/v1/traces")
    u = urlparse("http://localhost:6006")
    try:
        socket.create_connection((u.hostname, u.port), timeout=0.3).close()
        return "http://localhost:6006/v1/traces"
    except OSError:
        return None

MEMORIA = InMemorySpanExporter()
# openinference.project.name agrupa las trazas por proyecto en Phoenix
PROVEEDOR = TracerProvider(resource=Resource.create({"service.name": PROYECTO, "openinference.project.name": PROYECTO}))
PROVEEDOR.add_span_processor(SimpleSpanProcessor(MEMORIA))
PHOENIX = _phoenix_url()
if PHOENIX:
    from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
    cabeceras = {"authorization": f"Bearer {os.environ['PHOENIX_API_KEY']}"} if os.environ.get("PHOENIX_API_KEY") else {}
    PROVEEDOR.add_span_processor(BatchSpanProcessor(OTLPSpanExporter(endpoint=PHOENIX, headers=cabeceras)))
tracer = PROVEEDOR.get_tracer("curso.soporte")
print("Trazas en memoria: sí · Phoenix:", PHOENIX or "no detectado (todo funciona igual, solo sin la interfaz web)")

## 4 · Tacha los secretos antes de guardar
Una traza guarda **prompts completos**. Si un estudiante pega su key en el chat, la key termina en tu herramienta de observabilidad, en texto plano. Por eso **todo** atributo pasa por `redactar()` antes de guardarse. Es la misma regla de la clase 1.2, aplicada a la observabilidad.

In [ ]:
_SECRETOS = [
    (re.compile(r"nvapi-[A-Za-z0-9_\-]{8,}"), "nvapi-***"),
    (re.compile(r"(?i)bearer\s+[A-Za-z0-9._\-]{8,}"), "Bearer ***"),
    (re.compile(r"(?i)(cookie|token|password|contraseña)(\s*[:=]\s*)\S+"), r"\1\2***"),
]
MAX_ATRIBUTO = 4000     # los colectores recortan atributos enormes; mejor recortar nosotros

def redactar(valor):
    texto = valor if isinstance(valor, str) else json.dumps(valor, ensure_ascii=False, default=str)
    for patron, reemplazo in _SECRETOS:
        texto = patron.sub(reemplazo, texto)
    return texto if len(texto) <= MAX_ATRIBUTO else texto[:MAX_ATRIBUTO] + f"… (+{len(texto) - MAX_ATRIBUTO} caracteres)"

def poner(span, **atributos):
    """Pone atributos en un span, redactados. Los nombres con '_' se convierten a '.' (input_value -> input.value)."""
    for k, v in atributos.items():
        if v is None:
            continue
        clave = k.replace("__", "·").replace("_", ".").replace("·", "_")
        span.set_attribute(clave, v if isinstance(v, (int, float, bool)) else redactar(v))

print(redactar("mi key es nvapi-AbCdEf1234567890xyz y mi cookie=abc123def456"))

## 5 · Instrumenta las tres piezas del agente
- **`llamar_trazado`**: cada llamada al modelo es un span `LLM` con el modelo, los mensajes de entrada, la salida, `finish_reason` y los tokens.
- **`buscar_faq_trazado`**: la búsqueda es un span `RETRIEVER` con cada documento recuperado y su puntaje.
- **`estado_laboratorio`**: una herramienta simulada que consulta tu instancia de Brev. Le puedes **inyectar fallas** (`FALLAS`) para practicar.

In [ ]:
FALLAS = {"estado_laboratorio": None}     # None | "503" | "lento"

def _tokens(r, mensajes):
    u = getattr(r, "usage", None)
    if u and getattr(u, "total_tokens", 0):
        return u.prompt_tokens, u.completion_tokens
    # MOCK: la respuesta simulada no trae tokens; estimamos ≈ 4 caracteres por token
    entrada = sum(len(str(m.get("content") or "")) for m in mensajes) // 4
    m = r.choices[0].message
    salida = len((m.content or "") + json.dumps([t.function.arguments for t in (m.tool_calls or [])])) // 4
    return entrada, salida

def llamar_trazado(mensajes, herramientas=None, paso=0):
    with tracer.start_as_current_span(f"llm · paso {paso}") as s:
        poner(s, openinference_span_kind="LLM", llm_model__name=MODEL,
              input_value=mensajes[-3:], input_mime__type="application/json",
              llm_invocation__parameters={"temperature": 0, "tools": [t["function"]["name"] for t in herramientas or []]})
        t0 = time.time()
        if USE_MOCK:
            time.sleep(LATENCIA_MOCK)
        try:
            r = chat(mensajes, tools=herramientas, temperature=0)
        except Exception as e:
            s.record_exception(e)
            s.set_status(Status(StatusCode.ERROR, str(e).splitlines()[0]))
            raise
        m = r.choices[0].message
        entrada, salida = _tokens(r, mensajes)
        poner(s, output_value=m.content or [{"tool": t.function.name, "args": t.function.arguments} for t in m.tool_calls or []],
              llm_finish__reason=r.choices[0].finish_reason,
              llm_token__count_prompt=entrada, llm_token__count_completion=salida,
              llm_token__count_total=entrada + salida)
        LLAMADAS.append(("soporte-trazado", round(time.time() - t0, 2)))
        return r

def buscar_faq_trazado(consulta, k=3):
    with tracer.start_as_current_span("retriever · buscar_faq") as s:
        poner(s, openinference_span_kind="RETRIEVER", input_value=consulta, tool_name="buscar_faq")
        if not isinstance(consulta, str) or not consulta.strip():
            s.set_status(Status(StatusCode.ERROR, "consulta vacía"))
            return "ERROR: 'consulta' debe ser texto no vacío"
        hits = INDICE.buscar(consulta, k=max(1, min(int(k), 5)), minimo=MIN_SCORE)
        for i, h in enumerate(hits):
            poner(s, **{f"retrieval_documents_{i}_document_id": h["id"],
                        f"retrieval_documents_{i}_document_score": h["score"],
                        f"retrieval_documents_{i}_document_content": h["respuesta"][:300]})
        salida = armar_contexto(hits) if hits else "SIN_EVIDENCIA: ningún fragmento supera el umbral."
        poner(s, output_value=salida)
        return salida

def estado_laboratorio(instancia):
    """Herramienta simulada: el estado de tu instancia de Brev."""
    if not isinstance(instancia, str) or not instancia.strip():
        return "ERROR: 'instancia' debe ser el nombre de tu instancia (texto)"
    if FALLAS["estado_laboratorio"] == "503":
        raise ConnectionError("503 Service Unavailable desde la API de Brev (simulado)")
    if FALLAS["estado_laboratorio"] == "lento":
        time.sleep(1.5)
    return json.dumps({"instancia": instancia, "estado": "RUNNING", "reanudada_hace_min": 12,
                       "cookie_de_sesion": "vencida"}, ensure_ascii=False)

HERRAMIENTAS = [
    {"type": "function", "function": {
        "name": "buscar_faq",
        "description": "Busca en la FAQ del curso. Devuelve fragmentos con un id [archivo.md#pregunta] para citar, o SIN_EVIDENCIA.",
        "parameters": {"type": "object", "properties": {"consulta": {"type": "string"},
                                                        "k": {"type": "integer", "minimum": 1, "maximum": 5}},
                       "required": ["consulta"]}}},
    {"type": "function", "function": {
        "name": "estado_laboratorio",
        "description": "Consulta el estado de la instancia de Brev del estudiante (RUNNING, STOPPED...) y de su sesión del laboratorio.",
        "parameters": {"type": "object", "properties": {"instancia": {"type": "string"}}, "required": ["instancia"]}}},
]
IMPLEMENTACIONES = {"buscar_faq": buscar_faq_trazado, "estado_laboratorio": estado_laboratorio}

def ejecutar_herramienta(nombre, argumentos):
    """Un span TOOL por herramienta. Los errores vuelven al modelo como observación, y quedan en la traza."""
    with tracer.start_as_current_span(f"tool · {nombre}") as s:
        poner(s, openinference_span_kind="TOOL", tool_name=nombre, input_value=argumentos)
        try:
            args = json.loads(argumentos or "{}")
            salida = IMPLEMENTACIONES[nombre](**args)
        except Exception as e:
            s.record_exception(e)
            s.set_status(Status(StatusCode.ERROR, f"{type(e).__name__}: {e}"))
            salida = f"ERROR: {type(e).__name__}: {e}"
        if salida.startswith("ERROR") and s.status.status_code != StatusCode.ERROR:
            s.set_status(Status(StatusCode.ERROR, salida[:120]))
        poner(s, output_value=salida)
        return salida

print("Instrumentación lista: LLM, RETRIEVER y TOOL")

## 6 · El agente de soporte, con una traza por pregunta
Es el loop ReAct de la clase 2.2. Lo único nuevo: un span raíz `AGENT` envuelve todo, y cada paso queda como hijo.

In [ ]:
SISTEMA = """ROL: soporte-trazado
Eres el agente de soporte del curso Seguridad de agentes con OpenShell y NemoClaw.
- Si la duda es sobre la instancia o el laboratorio, primero usa estado_laboratorio con la instancia "mi-lab".
- Busca en la FAQ con buscar_faq. Cita los ids [archivo.md#...] tal cual aparecen; no inventes ids.
- Si buscar_faq devuelve SIN_EVIDENCIA, di "La FAQ no cubre esto" y no repitas la misma búsqueda.
- Responde en español con: Qué pasa, Qué hacer (pasos) y Fuente."""

def agente(pregunta, max_pasos=5):
    """Responde una pregunta y devuelve {'respuesta', 'traza', 'pasos'}."""
    with tracer.start_as_current_span("agente · soporte") as raiz:
        poner(raiz, openinference_span_kind="AGENT", input_value=pregunta)
        mensajes = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": pregunta}]
        respuesta = None
        for paso in range(1, max_pasos + 1):
            r = llamar_trazado(mensajes, HERRAMIENTAS, paso=paso)
            m = r.choices[0].message
            if not m.tool_calls:
                respuesta = m.content or ""
                break
            mensajes.append({"role": "assistant", "content": m.content or "",
                             "tool_calls": [{"id": t.id, "type": "function",
                                             "function": {"name": t.function.name, "arguments": t.function.arguments}}
                                            for t in m.tool_calls]})
            for t in m.tool_calls:
                salida = ejecutar_herramienta(t.function.name, t.function.arguments)
                mensajes.append({"role": "tool", "tool_call_id": t.id, "content": salida})
        if respuesta is None:
            respuesta = "(el agente llegó al límite de pasos sin responder)"
            raiz.set_status(Status(StatusCode.ERROR, f"límite de {max_pasos} pasos"))
        poner(raiz, output_value=respuesta, agente_pasos=paso)
        traza = format(raiz.get_span_context().trace_id, "032x")
    PROVEEDOR.force_flush()      # que Phoenix la reciba ya
    return {"respuesta": respuesta, "traza": traza, "pasos": paso}

REPORTE = "Me sale 401 en el laboratorio después de reanudar Brev"
r1 = agente(REPORTE)
print(r1["respuesta"])
print("\ntraza:", r1["traza"], "· pasos:", r1["pasos"])

## 7 · Lee la traza como árbol
`ver_traza` reconstruye el árbol con los spans guardados en memoria: tipo, nombre, duración, lo más importante de sus atributos y ✖ si terminó en error. Es lo mismo que muestra Phoenix, en texto.

In [ ]:
def spans_de(traza):
    return [s for s in MEMORIA.get_finished_spans() if format(s.context.trace_id, "032x") == traza]

def _resumen(s):
    a = s.attributes
    tipo = a.get("openinference.span.kind", "")
    if tipo == "LLM":
        return f"finish={a.get('llm.finish_reason')} · tokens={a.get('llm.token_count.total')}"
    if tipo == "RETRIEVER":
        docs = [f"{a.get(f'retrieval.documents.{i}.document.id', '').split('#')[0]}:{a.get(f'retrieval.documents.{i}.document.score')}"
                for i in range(5) if f"retrieval.documents.{i}.document.id" in a]
        return "docs=" + (", ".join(docs) or "ninguno")
    if tipo == "TOOL":
        return "args=" + str(a.get("input.value", ""))[:60]
    if tipo == "AGENT":
        return f"pasos={a.get('agente.pasos')}"
    return ""

def ver_traza(traza):
    spans = spans_de(traza)
    hijos = {}
    for s in spans:
        hijos.setdefault(s.parent.span_id if s.parent else None, []).append(s)
    def pinta(s, prefijo="", ultimo=True, raiz=False):
        ms = (s.end_time - s.start_time) / 1e6
        marca = "✖" if s.status.status_code == StatusCode.ERROR else "✔"
        rama = "" if raiz else ("└─ " if ultimo else "├─ ")
        tipo = s.attributes.get("openinference.span.kind", "")
        print(f"{prefijo}{rama}{marca} {tipo:9s} {s.name:28s} {ms:7.0f} ms  {_resumen(s)}")
        if s.status.status_code == StatusCode.ERROR and s.status.description:
            print(f"{prefijo}{'' if raiz else ('   ' if ultimo else '│  ')}   └ error: {s.status.description[:90]}")
        hs = sorted(hijos.get(s.context.span_id, []), key=lambda x: x.start_time)
        for i, h in enumerate(hs):
            pinta(h, prefijo + ("" if raiz else ("   " if ultimo else "│  ")), i == len(hs) - 1)
    for r in hijos.get(None, []):
        pinta(r, raiz=True)

ver_traza(r1["traza"])

## 8 · Métricas que salen de la traza
Con los spans ya no adivinas el costo: lo cuentas. Tiempo y tokens por tipo de paso.

In [ ]:
def metricas(traza):
    filas = {}
    for s in spans_de(traza):
        tipo = s.attributes.get("openinference.span.kind", "?")
        f = filas.setdefault(tipo, {"spans": 0, "ms": 0, "tokens": 0, "errores": 0})
        f["spans"] += 1
        f["ms"] += round((s.end_time - s.start_time) / 1e6)
        f["tokens"] += s.attributes.get("llm.token_count.total", 0) or 0
        f["errores"] += s.status.status_code == StatusCode.ERROR
    print(f"{'tipo':10s} {'spans':>5s} {'ms':>7s} {'tokens':>7s} {'errores':>7s}")
    for tipo, f in sorted(filas.items()):
        print(f"{tipo:10s} {f['spans']:5d} {f['ms']:7d} {f['tokens']:7d} {f['errores']:7d}")
    if USE_MOCK:
        print("(MOCK: tokens estimados ≈ caracteres/4; en vivo vienen de usage)")
    return filas

_ = metricas(r1["traza"])

## 9 · 🧯 Depura con la traza
Cuatro problemas típicos. En cada uno, **primero mira la traza** y luego lee la explicación.

### a) La herramienta falla
La API de Brev responde 503. ¿El agente lo oculta, o lo dice? ¿Dónde quedó el error?

In [ ]:
FALLAS["estado_laboratorio"] = "503"
ra = agente("¿Mi instancia de Brev está encendida? Me sale 401 en el laboratorio")
FALLAS["estado_laboratorio"] = None
print(ra["respuesta"], "\n")
ver_traza(ra["traza"])

El span `tool · estado_laboratorio` quedó con ✖ y el mensaje exacto (`ConnectionError: 503 ...`). La raíz terminó en ✔ porque el **harness** convirtió el error en observación y el agente siguió. Eso es correcto, **siempre que la respuesta lo diga**: revisa que diga "No pude verificar el estado de tu instancia". Sin traza, verías solo una respuesta incompleta y no sabrías por qué.

### b) El loop atascado
Una pregunta que la FAQ no cubre. El agente busca, no encuentra y vuelve a buscar **lo mismo**.

In [ ]:
def repeticiones(traza, minimo=2):
    """Llamadas a herramientas con los mismos argumentos repetidas en una traza."""
    cuenta = {}
    for s in spans_de(traza):
        if s.attributes.get("openinference.span.kind") == "TOOL":
            clave = (s.attributes.get("tool.name"), s.attributes.get("input.value"))
            cuenta[clave] = cuenta.get(clave, 0) + 1
    return {f"{n} {a}": c for (n, a), c in cuenta.items() if c >= minimo}

rb = agente("¿Me pueden mandar la factura de Brev en pesos?")
print(rb["respuesta"], "\n")
ver_traza(rb["traza"])
print("\nrepeticiones:", repeticiones(rb["traza"]))

La raíz termina con ✖ `límite de 5 pasos`: el tope de la clase 2.2 te salvó, pero gastaste 5 llamadas en nada. `repeticiones` muestra la causa: la misma búsqueda, con los mismos argumentos, en cada paso. **Arreglo:** en el harness, si una herramienta recibe los mismos argumentos dos veces seguidas, devuelve una observación como *"ya buscaste esto; responde con lo que tienes"*. En vivo, el modelo real puede rendirse antes; la traza te lo dice igual.

### c) La cita inventada
La respuesta "se ve bien" y trae citas. ¿Todas salieron de una herramienta?

In [ ]:
def citas_sin_respaldo(traza):
    """Ids citados en la respuesta final que NO aparecen en ninguna salida de herramienta de la traza."""
    spans = spans_de(traza)
    raiz = next(s for s in spans if s.parent is None)
    citadas = set(_ID.findall(raiz.attributes.get("output.value", "")))
    vistas = set()
    for s in spans:
        if s.attributes.get("openinference.span.kind") in ("TOOL", "RETRIEVER"):
            vistas |= set(_ID.findall(s.attributes.get("output.value", "")))
    return sorted(citadas - vistas)

rc = agente("¿Es cierto que el launchable de Brev trae GPU gratis para siempre?")
print(rc["respuesta"], "\n")
print("citas sin respaldo:", citas_sin_respaldo(rc["traza"]))
print("citas sin respaldo en la primera traza:", citas_sin_respaldo(r1["traza"]))

`brev.md#la-gpu-es-gratis-para-siempre` no salió de ninguna búsqueda: el modelo la inventó. Con la traza lo compruebas **con código**, no a ojo. Es la `revisar_citas` de la clase 3.3, ahora sobre la traza, y sirve igual para cualquier agente instrumentado.

### d) ¿Por qué tardó tanto?

In [ ]:
def mas_lentos(traza, n=3):
    spans = sorted(spans_de(traza), key=lambda s: s.end_time - s.start_time, reverse=True)
    return [(s.name, round((s.end_time - s.start_time) / 1e6)) for s in spans if s.parent is not None][:n]

FALLAS["estado_laboratorio"] = "lento"
rd = agente("Mi instancia de Brev no responde después de reanudar")
FALLAS["estado_laboratorio"] = None
print("los 3 spans más lentos (ms):", mas_lentos(rd["traza"]))

El cuello de botella es `tool · estado_laboratorio`, no el modelo. Sin traza, la reacción típica es "el modelo es lento, cambiemos de modelo". Con traza, arreglas lo que de verdad tarda.

## 10 · Phoenix y la auto-instrumentación
Hasta aquí instrumentaste **a mano**: entiendes qué es cada span. En proyectos reales también usas **auto-instrumentación**: una librería que envuelve el cliente de OpenAI y crea los spans `LLM` sola.

```bash
pip install arize-phoenix openinference-instrumentation-openai
phoenix serve            # interfaz en http://localhost:6006
```

Con tu key, la celda siguiente activa `OpenAIInstrumentor` sobre el mismo `PROVEEDOR`: los spans automáticos aparecen **dentro** de tu span `AGENT`. Sin key no hace nada (el `MockClient` no es el cliente de OpenAI).

En Phoenix: abre **Projects → soporte-nemoclaw**, elige una traza y recorre el árbol. A la derecha ves los atributos de cada span: los mismos que imprime `ver_traza`.

In [ ]:
if USE_MOCK:
    print("MOCK: la auto-instrumentación necesita el cliente real de OpenAI. Con tu key, esta celda la activa.")
else:
    _instala("openinference-instrumentation-openai", "openinference.instrumentation.openai")
    from openinference.instrumentation.openai import OpenAIInstrumentor
    OpenAIInstrumentor().instrument(tracer_provider=PROVEEDOR)
    r_auto = agente(REPORTE)
    ver_traza(r_auto["traza"])
    print("\nVerás spans 'ChatCompletion' (automáticos) dentro de tus spans 'llm · paso N'.")

### ¿Y NVIDIA tiene algo propio?
Sí, en dos niveles:

| Dónde | Herramienta de NVIDIA | Qué hace |
|---|---|---|
| Tu código de agente | **NeMo Agent Toolkit** (`nvidia-nat`) | Instrumenta flujos de agentes y exporta trazas a Phoenix, Langfuse, LangSmith, Weave u OTel con unas líneas de YAML. |
| El sandbox de NemoClaw | `nemoclaw <sandbox> logs`, `sessions export` y `openshell term` | Bitácoras del gateway, eventos de auditoría de OpenShell (bloqueos de política) y el historial de sesiones del agente. |
| NemoClaw con Deep Agents | `--observability` en el onboarding | Exporta trazas OTLP a un colector en el host, con el preset de red `observability-otlp-local`. |

Con el NeMo Agent Toolkit, mandar las trazas a Phoenix es configuración, no código:

```yaml
general:
  telemetry:
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://localhost:6006/v1/traces
        project: soporte-nemoclaw
```

Las trazas que hiciste aquí y las del toolkit hablan el mismo idioma (OTel + OpenInference): las ves en el mismo Phoenix.

## 11 · Una traza también es un dato sensible

In [ ]:
rs = agente("Me sale 401 en el laboratorio. Mi key es nvapi-ESTOESUNAKEYFALSA1234567890")
filtrada = any("ESTOESUNAKEYFALSA" in str(v) for s in spans_de(rs["traza"]) for v in s.attributes.values())
print("¿la key quedó en la traza?:", filtrada)
assert not filtrada, "¡La key llegó a la traza! Revisa que todos los atributos pasen por redactar()"
print(spans_de(rs["traza"])[-1].attributes["input.value"])

La key no llegó a la traza: `poner()` tacha todo con `redactar()`. Aun así, trata tus trazas como datos privados: contienen preguntas de estudiantes, rutas de archivos y salidas de herramientas. No publiques un Phoenix sin autenticación y no compartas trazas completas en un foro.

## 12 · Avance del proyecto
**Soporte NemoClaw v3.4: cada respuesta con su traza.** `exportar_traza` guarda la traza, con los secretos tachados, en JSON, lista para adjuntar a un reporte de error o comparar antes y después de un cambio.

In [ ]:
def exportar_traza(traza, carpeta="trazas"):
    pathlib.Path(carpeta).mkdir(exist_ok=True)
    datos = [{"nombre": s.name, "span_id": format(s.context.span_id, "016x"),
              "padre": format(s.parent.span_id, "016x") if s.parent else None,
              "ms": round((s.end_time - s.start_time) / 1e6), "estado": s.status.status_code.name,
              "error": s.status.description, "atributos": dict(s.attributes)}
             for s in sorted(spans_de(traza), key=lambda s: s.start_time)]
    ruta = pathlib.Path(carpeta) / f"{traza}.json"
    ruta.write_text(json.dumps(datos, ensure_ascii=False, indent=1), encoding="utf-8")
    return ruta

def soporte_nemoclaw(pregunta):
    r = agente(pregunta)
    problemas = {"citas_sin_respaldo": citas_sin_respaldo(r["traza"]), "repeticiones": repeticiones(r["traza"]),
                 "errores": [s.name for s in spans_de(r["traza"]) if s.status.status_code == StatusCode.ERROR]}
    return {**r, "problemas": problemas, "archivo": str(exportar_traza(r["traza"]))}

final = soporte_nemoclaw("¿Por qué mi notebook responde HTTP 401 cuando llamo al modelo?")
print(final["respuesta"])
print("\nproblemas:", final["problemas"])
print("traza guardada en:", final["archivo"])

## Resumen
- Una **traza** es el árbol de lo que pasó para responder una pregunta; un **span**, cada paso.
- Instrumentaste tu agente con **OpenTelemetry** y convenciones **OpenInference** (`AGENT`, `LLM`, `RETRIEVER`, `TOOL`).
- Con la traza encontraste una herramienta que falla, un loop atascado, una cita inventada y el paso lento, **con código**.
- **Tachas los secretos** antes de guardar: una traza es tan sensible como el chat.
- Phoenix y el **NeMo Agent Toolkit** consumen el mismo formato.

En el Módulo 4 tu agente se va a NemoClaw. En la clase 4.4 vas a trazar las llamadas a Hermes con este mismo `PROVEEDOR`.